# GraphConsis & BWGNN — Colab experiments

Train **GraphConsis** and **BWGNN** on YelpChi, Amazon, T-Finance, and T-Social.

**Easiest Colab setup:** zip the whole `Graph-Fraud-WK` folder (code + `dataset/`), upload that zip to Google Drive, then unpack it in the next cell.

Each experiment cell has a parameter dict you can edit. After training, artifacts are written under:

`notebook_runs/<Model>/<Dataset>/seed<seed>_<timestamp>_ep…/`

(config, metrics, split, predictions, checkpoint), then that folder is **zipped**. On Colab the zip downloads automatically.

**Runtime → Change runtime type → GPU** before installing packages.

T-Finance (~42M edges) and T-Social (multi-GB) are heavy. Prefer a GPU High-RAM runtime for those cells.

## 1. Unpack the repo zip from Google Drive

Zip **the project folder** on your computer (the one that contains `algorithms/` and `dataset/`), upload it to Drive, then set `DRIVE_REPO_ZIP` below.

Include in the zip:
- the code (`algorithms/`, `utils/`, this notebook, …)
- `dataset/` with either the extracted files (`YelpChi.mat`, `Amazon.mat`, `tfinance`, `tsocial`) **or** the inner zips (`YelpChi.zip`, …) — those inner zips are unpacked automatically

Exclude `.venv` (a Windows venv will not work on Colab and makes the zip huge). Exclude `results/` / `notebook_runs/` if you want a smaller upload.

Example: file at `My Drive / Graph-Fraud-WK.zip` → `DRIVE_REPO_ZIP = "Graph-Fraud-WK.zip"`

In [ ]:
import os
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

# Zip of the whole repo (code + dataset/) on Google Drive, relative to My Drive.
DRIVE_REPO_ZIP = "Graph-Fraud-WK.zip"
UNPACK_DEST = "/content" if IN_COLAB else str(Path.cwd() / "_unpacked_repo")


def _looks_like_repo(path):
    return (Path(path) / "algorithms" / "GraphConsis" / "supervised_train.py").is_file()


def _find_repo(starts):
    seen = set()
    for start in starts:
        start = Path(start)
        if not start.exists():
            continue
        queue = [start]
        if start.is_dir():
            queue.extend(sorted(p for p in start.iterdir() if p.is_dir()))
        for candidate in queue:
            key = str(candidate.resolve()) if candidate.exists() else str(candidate)
            if key in seen:
                continue
            seen.add(key)
            if _looks_like_repo(candidate):
                return candidate.resolve()
    dest = Path(UNPACK_DEST)
    if dest.is_dir():
        for hit in dest.rglob("supervised_train.py"):
            if hit.parent.name == "GraphConsis":
                return hit.parents[2].resolve()
    return None


def _resolve_zip(name):
    raw = str(name).strip().replace("\\", "/")
    p = Path(raw).expanduser()
    if p.is_file():
        return p.resolve()
    mydrive = Path("/content/drive/MyDrive")
    candidates = [mydrive / raw, Path("/content/drive") / raw, Path.cwd() / raw]
    if raw.lower().startswith("mydrive/"):
        candidates.insert(0, Path("/content/drive") / raw)
    for c in candidates:
        if c.is_file():
            return c.resolve()
    raise FileNotFoundError(
        "Repo zip not found: {!r}. After mounting Drive, set DRIVE_REPO_ZIP "
        "relative to My Drive (e.g. 'Graph-Fraud-WK.zip').".format(name)
    )


if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

repo = _find_repo([Path.cwd(), *Path.cwd().parents, Path(UNPACK_DEST)])

if repo is None:
    if not str(DRIVE_REPO_ZIP).strip():
        raise FileNotFoundError("Set DRIVE_REPO_ZIP to your Drive zip of the repo.")
    zip_path = _resolve_zip(DRIVE_REPO_ZIP)
    dest = Path(UNPACK_DEST)
    dest.mkdir(parents=True, exist_ok=True)
    print("Unpacking", zip_path, "->", dest)
    unzip_bin = shutil.which("unzip")
    if unzip_bin:
        subprocess.check_call([unzip_bin, "-o", "-q", str(zip_path), "-d", str(dest)])
    else:
        with zipfile.ZipFile(zip_path) as zf:
            zf.extractall(dest)
    repo = _find_repo([dest])

if repo is None:
    raise FileNotFoundError(
        "Could not find algorithms/GraphConsis after unzip. Zip the Graph-Fraud-WK "
        "folder itself (the one that contains algorithms/ and dataset/)."
    )

os.chdir(repo)
if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))
print("Repo root:", Path.cwd().resolve())
print("Colab:", IN_COLAB)

## 2. Install packages (Colab)

Colab already ships CUDA PyTorch. Install PyG and sympy only — do **not** `pip install dgl` (broken on many Colab / Windows Python builds). T-Finance / T-Social load without DGL.

Re-run this cell after a runtime restart.

In [ ]:
import torch

print("torch", torch.__version__, "cuda", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")

%pip install -q torch-geometric sympy

import torch_geometric
import sympy
print("torch-geometric", torch_geometric.__version__, "sympy", sympy.__version__)

## 3. Check datasets

If `dataset/` inside the repo zip already has `YelpChi.mat`, `Amazon.mat`, `tfinance`, and `tsocial`, this cell only verifies them.

If you zipped the **inner** archives (`YelpChi.zip`, `Amazon.zip`, `tfinance.zip`, `tsocial.zip`) instead, they are unpacked here.

Leave `DRIVE_DATASET_DIR` empty unless the graphs live in a **separate** Drive folder (not inside the repo zip).

In [ ]:
from utils.notebook_runner import (
    dataset_dir,
    missing_dataset_files,
    prepare_datasets_from_drive,
    unzip_if_needed,
)

# Only if the graphs are NOT already under the unzipped repo's dataset/.
# Relative to My Drive, e.g. "GraphFraud/zips"
DRIVE_DATASET_DIR = ""

ddir = dataset_dir()
unzip_if_needed(ddir)

if str(DRIVE_DATASET_DIR).strip():
    missing = prepare_datasets_from_drive(DRIVE_DATASET_DIR, dest=ddir)
else:
    missing = missing_dataset_files(ddir)

print("dataset/:", ddir.resolve())
for p in sorted(ddir.iterdir()):
    if p.is_file() and not p.name.lower().endswith(".zip"):
        print("  {:<24} {:8.1f} MB".format(p.name, p.stat().st_size / 1e6))
if missing:
    print("Still missing:", missing)
    print("Put those files (or their .zip) in dataset/ inside the repo zip, or set DRIVE_DATASET_DIR.")
else:
    print("All four dataset files are present.")

## 4. Run helpers

Edit the dict in a cell below, then run that cell. Each run gets its own folder plus a zip.

Set `download=False` if you do not want the Colab download dialog every time (zips are still written next to the folder).

In [ ]:
import torch
from utils.notebook_runner import (
    notebook_runs_dir,
    run_bwgnn,
    run_graphconsis,
    zip_all_notebook_runs,
)

print("Outputs:", notebook_runs_dir().resolve())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")

---
## GraphConsis

Tunable keys: `epochs`, `learning_rate`, `dim_1`, `batch_size`, `dropout`, `weight_decay`, `context_dim`, `sigmoid`, `samples_1`, `samples_2`, `test_size`, `seed`, `gpu`.

Folder: `notebook_runs/GraphConsis/<Dataset>/seed<seed>_<tag>/`

In [ ]:
# GraphConsis · YelpChi — edit params, then run this cell
GC_YELPCHI = dict(
    epochs=50,
    learning_rate=0.01,
    dim_1=128,
    batch_size=512,
    dropout=0.0,
    weight_decay=0.0,
    context_dim=0,
    sigmoid=True,
    samples_1=25,
    samples_2=10,
    test_size=0.2,
    seed=123,
    gpu="0",
)

gc_yelpchi = run_graphconsis("yelpchi", download=True, **GC_YELPCHI)
gc_yelpchi["metrics"]

In [ ]:
# GraphConsis · Amazon — edit params, then run this cell
GC_AMAZON = dict(
    epochs=50,
    learning_rate=0.01,
    dim_1=128,
    batch_size=512,
    dropout=0.0,
    weight_decay=0.0,
    context_dim=0,
    sigmoid=True,
    samples_1=25,
    samples_2=10,
    test_size=0.4,
    seed=123,
    gpu="0",
)

gc_amazon = run_graphconsis("amazon", download=True, **GC_AMAZON)
gc_amazon["metrics"]

In [ ]:
# GraphConsis · T-Finance — 42M edges; GPU recommended
GC_TFINANCE = dict(
    epochs=50,
    learning_rate=0.01,
    dim_1=128,
    batch_size=512,
    dropout=0.0,
    weight_decay=0.0,
    context_dim=0,
    sigmoid=True,
    samples_1=25,
    samples_2=10,
    test_size=0.4,
    seed=123,
    gpu="0",
)

gc_tfinance = run_graphconsis("tfinance", download=True, **GC_TFINANCE)
gc_tfinance["metrics"]

In [ ]:
# GraphConsis · T-Social — very large; smaller batch / fewer epochs by default
GC_TSOCIAL = dict(
    epochs=20,
    learning_rate=0.01,
    dim_1=64,
    batch_size=256,
    dropout=0.0,
    weight_decay=0.0,
    context_dim=0,
    sigmoid=True,
    samples_1=10,
    samples_2=5,
    test_size=0.4,
    seed=123,
    gpu="0",
)

gc_tsocial = run_graphconsis("tsocial", download=True, **GC_TSOCIAL)
gc_tsocial["metrics"]

---
## BWGNN

Tunable keys: `epochs`, `learning_rate`, `hid_dim`, `order`, `weight_decay`, `train_ratio` (overrides `test_size`), `test_size`, `seed`, `gpu`.

Paper defaults: `train_ratio=0.4`; T-Social uses `hid_dim=10`, `order=5`.

Folder: `notebook_runs/BWGNN/<dataset>/seed<seed>_<tag>/`

In [ ]:
# BWGNN · YelpChi — edit params, then run this cell
BW_YELPCHI = dict(
    epochs=100,
    learning_rate=0.01,
    hid_dim=64,
    order=2,
    weight_decay=0.0,
    train_ratio=None,
    test_size=0.2,
    seed=123,
    gpu="0",
)

bw_yelpchi = run_bwgnn("yelpchi", download=True, **BW_YELPCHI)
bw_yelpchi["metrics"]

In [ ]:
# BWGNN · Amazon — edit params, then run this cell
BW_AMAZON = dict(
    epochs=100,
    learning_rate=0.01,
    hid_dim=64,
    order=2,
    weight_decay=0.0,
    train_ratio=0.4,
    seed=123,
    gpu="0",
)

bw_amazon = run_bwgnn("amazon", download=True, **BW_AMAZON)
bw_amazon["metrics"]

In [ ]:
# BWGNN · T-Finance — paper: hid 64, order 2, 40% train
BW_TFINANCE = dict(
    epochs=100,
    learning_rate=0.01,
    hid_dim=64,
    order=2,
    weight_decay=0.0,
    train_ratio=0.4,
    seed=123,
    gpu="0",
)

bw_tfinance = run_bwgnn("tfinance", download=True, **BW_TFINANCE)
bw_tfinance["metrics"]

In [ ]:
# BWGNN · T-Social — paper: hid 10, order 5; needs a large GPU
BW_TSOCIAL = dict(
    epochs=100,
    learning_rate=0.01,
    hid_dim=10,
    order=5,
    weight_decay=0.0,
    train_ratio=0.4,
    seed=123,
    gpu="0",
)

bw_tsocial = run_bwgnn("tsocial", download=True, **BW_TSOCIAL)
bw_tsocial["metrics"]

## Download every notebook run

Optional: one zip of `notebook_runs/` after you have finished several cells.

In [ ]:
zip_all_notebook_runs(download=True)